# RiskNet – Data Preprocessing & Relationship Augmentation

## Objective

This notebook prepares the Bank Account Fraud (BAF) dataset for RiskNet.

The preprocessing pipeline performs the following tasks:

- Load the raw dataset from Databricks Volume.
- Analyze dataset quality.
- Clean and validate records.
- Generate relationship attributes required for graph-based fraud investigation.
- Validate the generated relationships.
- Save the curated dataset as a Delta Table for downstream applications.

The final curated dataset will be exported once and used by PostgreSQL and Neo4j in the RiskNet application.

Output:
• risknet_curated (Delta)
• Export CSV

## Why Databricks?

Although preprocessing could be performed using Python locally, Databricks was selected to implement a production-style data engineering workflow.

Reasons:

- Centralized storage of raw datasets.
- Reproducible preprocessing pipeline.
- Delta Tables provide reliable curated datasets.
- Easy handling of large datasets.
- Clear separation between data engineering and application development.

Databricks is used only during preprocessing. After preprocessing, the curated dataset is exported once and consumed by PostgreSQL and Neo4j.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

## Step 1 – Load Raw Dataset

The raw Bank Account Fraud dataset is stored inside a Databricks Volume.

The objective of this step is to load the dataset into Spark for distributed processing.

In [0]:
file_path = "/Volumes/risknet_catalog/risknet/risknet_volume/Base_ds3.csv"

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(file_path)
)

display(df.limit(10))

fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0.3,0.986506310633034,-1,25,40,0.0067353870811739,102.45371092469456,AA,1059,13096.035018400871,7850.955007125409,6742.080561007602,5,5,CB,163,1,BC,0,1,9,0,1500.0,0,INTERNET,16.224843433978073,linux,1,1,0,0
0,0.8,0.6174260062650061,-1,89,20,0.010095097878573,-0.8495509687507287,AD,1658,9223.283430930423,5745.251480643791,5941.6648588359885,3,18,CA,154,1,BC,1,1,2,0,1500.0,0,INTERNET,3.36385371062431,other,1,1,0,0
0,0.8,0.9967070206409232,9,14,40,0.0123163495250501,-1.4903855214855013,AB,1095,4471.472148765561,5471.988958014066,5992.555113248597,15,11,CA,89,1,BC,0,1,30,0,200.0,0,INTERNET,22.73055923496224,windows,0,1,0,0
0,0.6000000000000001,0.4750999462380287,11,14,30,0.0069908306305029,-1.8631006767118188,AB,3483,14431.993621381194,6755.34447929092,5970.336830507939,11,13,CA,90,1,BC,0,1,1,0,200.0,0,INTERNET,15.215816122068803,linux,1,1,0,0
0,0.9,0.8423068370138775,-1,29,40,5.742625847255586,47.152497798787614,AA,2339,7601.511579253147,5124.046929628144,5940.734211620649,1,6,CA,91,0,BC,1,1,26,0,200.0,0,INTERNET,3.743047928033851,other,0,1,0,0
0,0.6000000000000001,0.2948404635987846,-1,369,30,0.024231885459018,-1.232556198255754,AD,1204,11556.955513727777,7506.951275553085,6482.924037404599,705,5,CB,134,1,BE,1,1,30,0,200.0,0,INTERNET,6.987315503150867,linux,1,1,0,0
0,0.2,0.7730854651787792,22,4,40,0.0069193596555432,-0.5446764953031848,AB,1998,11723.993606010376,7864.277143635152,6338.799156196561,28,8,CA,72,1,BC,1,1,1,0,200.0,0,INTERNET,28.19992278516053,x11,1,1,0,0
0,0.8,0.153880432777335,-1,103,40,0.0451218390683624,-1.1011844334405805,AB,1548,4999.555800752307,4526.861667090668,6426.790816709525,6,7,CA,163,0,BE,1,1,25,1,200.0,0,INTERNET,11.23426418538036,other,1,1,0,0
0,0.3,0.5236546591292989,21,2,30,0.0352056883298541,-0.9557372072420892,AB,1781,6979.9940022111805,4335.68534636426,6624.957941677998,2,10,CA,35,0,BC,1,0,2,0,200.0,0,INTERNET,5.329386683741856,other,1,1,0,0
0,0.8,0.8344745881003516,-1,134,20,0.0172445227162768,-1.3563931999585843,AD,3113,7549.992085884615,6273.922109554705,6312.998834706559,14,20,CA,201,1,BD,1,1,15,0,1500.0,0,INTERNET,4.103969778315406,other,1,1,0,0


## Step 2 – Dataset Profiling

Before modifying the dataset, its quality is assessed.

The following checks are performed:

- Number of records
- Number of columns
- Data types
- Missing values
- Duplicate records
- Fraud distribution

Understanding the dataset before preprocessing ensures that augmentation is performed on clean and validated data.

In [0]:
print(f"Rows    : {df.count()}")
print(f"Columns : {len(df.columns)}")

Rows    : 1000000
Columns : 32


In [0]:
df.printSchema()

root
 |-- fraud_bool: integer (nullable = true)
 |-- income: double (nullable = true)
 |-- name_email_similarity: double (nullable = true)
 |-- prev_address_months_count: integer (nullable = true)
 |-- current_address_months_count: integer (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- days_since_request: double (nullable = true)
 |-- intended_balcon_amount: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- zip_count_4w: integer (nullable = true)
 |-- velocity_6h: double (nullable = true)
 |-- velocity_24h: double (nullable = true)
 |-- velocity_4w: double (nullable = true)
 |-- bank_branch_count_8w: integer (nullable = true)
 |-- date_of_birth_distinct_emails_4w: integer (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- credit_risk_score: integer (nullable = true)
 |-- email_is_free: integer (nullable = true)
 |-- housing_status: string (nullable = true)
 |-- phone_home_valid: integer (nullable = true)
 |-- phone_mobil

In [0]:
display(df.describe())

summary,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
count,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000,1000000
mean,0.011029,0.5626956000000318,0.49369409496312905,16.718568,86.587867,33.68908,1.025705230995809,8.661498537172443,null,1572.692049,5665.296604795234,4769.781964962302,4856.324015811875,184.361849,9.503544,null,130.989595,0.529886,null,0.417077,0.889676,10.839303,0.222988,515.85101,0.025242,null,7.544940201289974,null,0.576947,1.018312,0.0,3.288674
stddev,0.104438364916213,0.2903426011446931,0.2891247999953665,44.04623003277075,88.40659910389651,12.025798658443993,5.381834634593085,20.23615460624682,null,1005.3745649573393,3009.380665329911,1479.212612015712,919.8439342098833,459.6253290167275,5.033791888738029,null,69.68181240676046,0.4991062773709385,null,0.49307607850616647,0.31329333407641025,12.116874526578183,0.4162505556999595,487.55990185477486,0.15685938301824995,null,8.033106369073154,null,0.49404392848248146,0.18076145421716247,0.0,2.2099941642000336
min,0,0.1,1.4345504845275636E-6,-1,-1,10,4.036859788721786E-9,-15.530554840076814,AA,1,-170.60307235124628,1300.3073144849477,2825.748405284728,0,0,CA,-170,0,BA,0,0,-1,0,190.0,0,INTERNET,-1.0,linux,0,-1,0,0
max,1,0.9,0.9999993177937188,383,428,90,78.45690383509861,112.9569276953714,AE,6700,16715.565404174275,9506.896596111665,6994.764200834217,2385,39,CG,389,1,BG,1,1,32,1,2100.0,1,TELEAPP,85.89914319274027,x11,1,2,0,7


In [0]:
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(null_counts)

fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
duplicates = df.count() - df.dropDuplicates().count()

print("Duplicate Rows:", duplicates)

Duplicate Rows: 0


In [0]:
display(
    df.groupBy("fraud_bool").count()
)

fraud_bool,count
0,988971
1,11029


## Profiling Inference

The initial profiling shows that the BAF dataset is already well-structured and suitable for further processing.

### Observations

- The dataset contains **1,000,000 records** and **32 attributes**, making it representative of a large-scale banking dataset.
- No missing values were found in any column.
- No duplicate records were detected.
- All columns were correctly inferred with appropriate data types.
- The fraud distribution is highly imbalanced:
  - **988,971 legitimate records (98.90%)**
  - **11,029 fraudulent records (1.10%)**
- The dataset contains behavioural and risk-related features that can be used during relationship augmentation and fraud prioritization, such as transaction velocity, credit risk score, device operating system, and payment type.

### Conclusion

Since the dataset is already clean, extensive data cleaning is not required. The preprocessing phase will therefore focus on **data validation, standardization, and relationship augmentation** rather than correcting data quality issues. This preserves the integrity of the original BAF dataset while preparing it for graph-based fraud investigation in RiskNet.

## Step 3 – Data Cleaning

Operations:

- Remove duplicates
- Handle nulls
- Fix data types
- Validate ranges
- Standardize categorical values

Purpose: Create a reliable base dataset.

In [0]:
# Remove duplicate records
clean_df = df.dropDuplicates()

# Remove records with null values
clean_df = clean_df.dropna()

print("Rows after cleaning:", clean_df.count())

Rows after cleaning: 1000000


### Validate Numerical Features

Before augmentation, important numerical attributes are validated to ensure they lie within expected ranges defined by the dataset.

In [0]:
numeric_columns = [
    "income",
    "customer_age",
    "credit_risk_score",
    "velocity_6h",
    "velocity_24h",
    "velocity_4w",
    "proposed_credit_limit"
]

display(clean_df.select(numeric_columns).describe())

summary,income,customer_age,credit_risk_score,velocity_6h,velocity_24h,velocity_4w,proposed_credit_limit
count,1000000,1000000,1000000,1000000,1000000,1000000,1000000
mean,0.5626956000000454,33.68908,130.989595,5665.296604795235,4769.781964962294,4856.324015811857,515.85101
stddev,0.29034260114469296,12.025798658443993,69.68181240676046,3009.380665329911,1479.2126120157122,919.8439342098839,487.55990185477503
min,0.1,10,-170,-170.60307235124628,1300.3073144849477,2825.748405284728,190.0
max,0.9,90,389,16715.565404174275,9506.896596111665,6994.764200834217,2100.0


In [0]:
# Standardize Categorical Features
from pyspark.sql.functions import trim, col

categorical_columns = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os"
]

for c in categorical_columns:
    clean_df = clean_df.withColumn(c, trim(col(c)))

## Cleaning Summary

The BAF dataset required minimal cleaning because it already satisfied basic data quality requirements.

Cleaning results:

- Duplicate records removed: **0**
- Missing records removed: **0**
- Data types validated
- Numerical features verified
- Categorical values standardized

The cleaned dataset is now ready for relationship augmentation.

## Step 4 – Relationship Analysis

Analyze existing columns:

- device_os
- device_distinct_emails_8w
- fraud_bool
- source
- velocity
- credit_risk_score

Determine:

- Which features indicate shared devices?
- Which indicate suspicious behavior?

Purpose: Build augmentation rules from existing data rather than randomly.

The original BAF dataset does not contain explicit relationship identifiers such as Device ID, IP Address, or Applicant ID.

Before generating these attributes, the existing features are analyzed to identify behavioural patterns that can guide realistic relationship generation.

The objective is to derive augmentation rules from the dataset rather than assigning relationships randomly.

### Feature Analysis – fraud_bool

The `fraud_bool` column is the target variable of the BAF dataset.

It indicates whether an application is fraudulent or legitimate.

- 0 → Legitimate application
- 1 → Fraudulent application

Although RiskNet is not a fraud detection system, this label is valuable during preprocessing because it helps generate realistic relationship attributes.

Instead of assigning shared devices or IP addresses randomly, fraudulent applications can be given a higher probability of sharing infrastructure, creating realistic fraud networks for graph-based investigation.

In [0]:
display(
    clean_df.groupBy("fraud_bool")
            .count()
            .orderBy("fraud_bool")
)

fraud_bool,count
0,988971
1,11029


#### Inference

The dataset is highly imbalanced, with approximately **98.9% legitimate** and **1.1% fraudulent** applications.

This distribution reflects real-world banking datasets where fraud cases are rare.

For augmentation, this feature will guide relationship generation:

- Legitimate applicants will mostly receive unique devices and IP addresses.
- Fraudulent applicants will have a higher probability of sharing generated devices, IP addresses, and other identifiers.

This creates realistic relationship clusters without altering the original fraud labels.

### Feature Analysis – device_os

The `device_os` column represents the operating system used by the applicant during the account opening process.

Although it does not uniquely identify a device, it provides useful contextual information for relationship generation.

Applicants using the same operating system may realistically share generated devices, but device operating system alone is not sufficient to establish a relationship.

During augmentation, the generated Device ID will always be associated with a single operating system to maintain consistency across the dataset.

In [0]:
display(
    clean_df.groupBy("device_os")
            .count()
            .orderBy(F.desc("count"))
)

device_os,count
other,342728
linux,332712
windows,263506
macintosh,53826
x11,7228


In [0]:
display(
    clean_df.groupBy("device_os", "fraud_bool")
            .count()
            .orderBy("device_os", "fraud_bool")
)

device_os,fraud_bool,count
linux,0,330997
linux,1,1715
macintosh,0,53074
macintosh,1,752
other,0,340754
other,1,1974
windows,0,256999
windows,1,6507
x11,0,7147
x11,1,81


#### Inference

The dataset contains multiple operating systems used during account creation.

The fraud distribution across operating systems helps determine whether certain platforms appear more frequently in fraudulent applications.

However, the operating system itself is **not** used to classify fraud.

Instead, it serves as supporting information during relationship augmentation. When multiple applicants are assigned the same generated Device ID, they will also share the same operating system, ensuring realistic and internally consistent device profiles.

### Feature Analysis – device_distinct_emails_8w

The `device_distinct_emails_8w` feature represents the number of distinct email addresses associated with the same device during the previous eight weeks.

Unlike `device_os`, this feature directly reflects device-sharing behaviour.

A higher value indicates that a device has been linked to multiple email identities, which may suggest shared usage or suspicious activity.

This makes the feature highly relevant for designing realistic Device ID generation rules.

In [0]:
display(
    clean_df.groupBy("device_distinct_emails_8w")
            .count()
            .orderBy("device_distinct_emails_8w")
)

device_distinct_emails_8w,count
-1,359
0,6272
1,968067
2,25302


In [0]:
display(
    clean_df.groupBy("device_distinct_emails_8w", "fraud_bool")
            .count()
            .orderBy("device_distinct_emails_8w", "fraud_bool")
)

device_distinct_emails_8w,fraud_bool,count
-1,0,355
-1,1,4
0,0,6121
0,1,151
1,0,958228
1,1,9839
2,0,24267
2,1,1035


### Feature Analysis – Transaction Velocity

The BAF dataset provides three velocity features that measure applicant activity over different time periods:

- `velocity_6h`
- `velocity_24h`
- `velocity_4w`

These features describe behavioural intensity rather than relationships.

Higher velocity may indicate unusually frequent account activity, while lower velocity represents normal behaviour.

Although these features do not directly identify shared devices or IP addresses, they are useful for generating realistic fraud patterns and later fraud prioritization.

In [0]:
display(
    clean_df.select(
        "velocity_6h",
        "velocity_24h",
        "velocity_4w"
    ).describe()
)

summary,velocity_6h,velocity_24h,velocity_4w
count,1000000,1000000,1000000
mean,5665.296604795235,4769.781964962294,4856.324015811857
stddev,3009.380665329911,1479.2126120157122,919.8439342098839
min,-170.60307235124628,1300.3073144849477,2825.748405284728
max,16715.565404174275,9506.896596111665,6994.764200834217


In [0]:
display(
    clean_df.groupBy("fraud_bool")
            .agg(
                F.avg("velocity_6h").alias("Avg Velocity 6h"),
                F.avg("velocity_24h").alias("Avg Velocity 24h"),
                F.avg("velocity_4w").alias("Avg Velocity 4w")
            )
)

fraud_bool,Avg Velocity 6h,Avg Velocity 24h,Avg Velocity 4w
0,5670.664987564237,4771.528848831136,4857.444566420302
1,5183.913444449583,4613.138798160986,4755.844184841342


#### Inference

The average velocity values for legitimate and fraudulent applications are broadly similar, with only minor differences.

This indicates that transaction velocity alone is not a strong discriminator between fraudulent and legitimate applications in the BAF dataset.

Therefore, these features are **not used to generate relationship attributes** such as shared Device IDs or IP addresses.

Instead, they are retained as behavioural features that can provide additional context to the Isolation Forest model during fraud prioritization.

### Feature Analysis – credit_risk_score

The `credit_risk_score` feature represents the applicant's credit risk assessment provided in the BAF dataset.

This feature is not a relationship identifier because applicants with similar credit scores are not necessarily connected.

However, it provides valuable risk context that can support fraud prioritization after relationships have been established.

Therefore, it is retained as a behavioural attribute rather than being used to generate shared entities.

In [0]:
display(
    clean_df.select("credit_risk_score").describe()
)

summary,credit_risk_score
count,1000000
mean,130.989595
stddev,69.68181240676046
min,-170
max,389


In [0]:
display(
    clean_df.groupBy("fraud_bool")
            .agg(
                F.avg("credit_risk_score").alias("Average Credit Risk Score"),
                F.min("credit_risk_score").alias("Minimum"),
                F.max("credit_risk_score").alias("Maximum")
            )
)

fraud_bool,Average Credit Risk Score,Minimum,Maximum
0,130.4699035664342,-170,389
1,177.59035270650105,-97,378


#### Inference

The average credit risk scores of legitimate and fraudulent applications show some variation, but the score alone does not establish relationships between applicants.

Since RiskNet focuses on graph-based investigation, this feature is not used to generate Device IDs, IP addresses, or other shared entities.

Instead, it is retained as contextual information that can contribute to fraud prioritization alongside graph-derived features.

## Feature Analysis Summary

- **fraud_bool** – Guides fraud-aware relationship generation by assigning a higher probability of shared entities to fraudulent records.
- **device_os** – Ensures consistency by associating each generated Device ID with a single operating system.
- **device_distinct_emails_8w** – Serves as the primary reference for generating realistic shared Device IDs.
- **velocity_6h, velocity_24h, velocity_4w** – Retained as behavioural context for ML-based alert prioritization; not used for relationship generation.
- **credit_risk_score** – Preserved as applicant-level risk context for prioritization rather than relationship creation.
- **source** – Retained as application channel information for investigation context; not used to generate relationships.

### Conclusion

The analysis identified **`fraud_bool`** and **`device_distinct_emails_8w`** as the primary drivers for relationship augmentation, while the remaining features are preserved as contextual information to support investigation and ML-based fraud prioritization.

In [0]:
display(
    clean_df.select(
        F.count("*").alias("Rows")
    )
)

Rows
1000000


In [0]:
display(
    clean_df.groupBy()
    .count()
)

count
1000000


In [0]:
display(
    clean_df
    .groupBy(clean_df.columns)
    .count()
    .filter(F.col("count") > 1)
)

fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month,count


## Step 4 – Relationship Augmentation

### Objective

The original BAF dataset contains behavioural and risk-related attributes but does not explicitly represent relationships between applicants or investigation alerts.

Since RiskNet is a graph-based fraud investigation system, realistic entities and relationships are required to simulate fraud networks that can be visualized, explored, and investigated.

This augmentation stage generates the following investigation entities:

- ApplicantID
- DeviceID
- IP Address
- Registration Timestamp
- AlertID

In a real banking environment, Device IDs, IP addresses, and registration timestamps are automatically captured by the onboarding infrastructure and associated with each application. Since the BAF dataset omits these attributes due to privacy and anonymization constraints, they are synthetically generated in this project using fraud-aware augmentation rules to simulate realistic system-captured metadata and enable graph-based fraud investigation.

The generation process follows fraud-aware rules so that fraudulent applicants are more likely to share devices and IP addresses, producing realistic fraud clusters while maintaining plausible distributions.

Each applicant is assigned a unique ApplicantID, while only suspicious applications receive an AlertID, representing alerts generated by an upstream fraud detection engine. These alerts act as the entry point for investigation within RiskNet.

The augmented dataset will later be imported into PostgreSQL and Neo4j, where alerts, applicants, devices, and IP addresses are connected to construct an investigation graph for fraud analysis.

## Relationship Generation Strategy

The relationship augmentation process follows the design principles below to create a realistic fraud investigation dataset while preserving the characteristics of the original BAF dataset.

- Every application is assigned a unique **ApplicantID**.
- **AlertIDs** are generated for all fraudulent applications and a configurable random sample of legitimate applications (`FALSE_POSITIVE_ALERT_RATE`) to simulate real-world upstream fraud detection systems that produce both true-positive and false-positive alerts.
- DeviceIDs are generated using distributed PySpark DataFrame transformations with configurable sharing probabilities and controlled fraud cluster sizes.
- Applicants sharing a DeviceID always have the same `device_os` to maintain realistic device consistency.
- **IP Addresses** are generated independently with controlled reuse to simulate shared networks and potential fraud rings.
- **Registration Timestamps** are distributed over a realistic time period, with related applicants occurring within similar time windows.
- Fraudulent applicants have a higher probability of sharing devices and IP addresses than legitimate applicants, resulting in realistic investigation clusters.
- The generated entities collectively enable construction of an investigation graph linking **Alerts → Applicants → Devices → IP Addresses** in Neo4j.

%md
### Generate ApplicantID

Each application is assigned a unique **ApplicantID**, which serves as the primary identifier throughout the RiskNet pipeline.

Unlike Device IDs or IP addresses, ApplicantIDs are unique and are never shared between applications. They provide a stable reference for linking alerts, relationships, investigation results, and exported graph data across PostgreSQL and Neo4j.

A sequential identifier with the prefix **APP** is generated for every record.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.orderBy(F.monotonically_increasing_id())

aug_df = (
    clean_df
    .withColumn("ApplicantSeq", F.row_number().over(window_spec))
    .withColumn(
        "ApplicantID",
        F.concat(
            F.lit("APP"),
            F.lpad(F.col("ApplicantSeq"), 8, "0")
        )
    )
    .drop("ApplicantSeq")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    aug_df.select(
        F.count("*").alias("Rows"),
        F.countDistinct("ApplicantID").alias("UniqueApplicantIDs")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Rows,UniqueApplicantIDs
1000000,1000000


In [0]:
display(
    aug_df.select("ApplicantID").limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID
APP00000001
APP00000002
APP00000003
APP00000004
APP00000005
APP00000006
APP00000007
APP00000008
APP00000009
APP00000010


Each application now has a unique ApplicantID that serves as the primary key for downstream relationship generation, graph construction, and investigation workflows.

### Generate AlertID

RiskNet investigates alerts produced by an upstream fraud detection engine rather than every application.

To simulate this workflow, AlertIDs are assigned to:

- All fraudulent applications (`fraud_bool = 1`).
- A configurable random sample of legitimate applications (`FALSE_POSITIVE_ALERT_RATE`) representing false-positive alerts.

Applications without an AlertID are considered low risk and are not forwarded for investigation.

Each AlertID uniquely identifies an investigation event and serves as the entry point for graph-based fraud analysis and case management.

In [0]:
# Configurable Parameters
FALSE_POSITIVE_ALERT_RATE = 0.10

In [0]:
from pyspark.sql import functions as F

# Randomly select false-positive alerts
aug_df = aug_df.withColumn(
    "is_false_positive_alert",
    (F.col("fraud_bool") == 0) &
    (F.rand(seed=42) < FALSE_POSITIVE_ALERT_RATE)
)

# Applications forwarded to RiskNet
aug_df = aug_df.withColumn(
    "GenerateAlert",
    (F.col("fraud_bool") == 1) |
    F.col("is_false_positive_alert")
)

# Generate AlertID
aug_df = aug_df.withColumn(
    "AlertID",
    F.when(
        F.col("GenerateAlert"),
        F.concat(
            F.lit("ALT"),
            F.lpad(F.monotonically_increasing_id().cast("string"), 8, "0")
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    aug_df.groupBy("fraud_bool", "GenerateAlert")
          .count()
          .orderBy("fraud_bool", "GenerateAlert")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


fraud_bool,GenerateAlert,count
0,false,890147
0,true,98824
1,true,11029


Fraudulent applications always generate investigation alerts, while a configurable proportion of legitimate applications also generate alerts to simulate false positives produced by real-world fraud detection systems. This creates a realistic investigation queue for RiskNet, where analysts review both genuine fraud cases and benign applications before reaching a final decision.

### Generate DeviceID

Device IDs are one of the strongest relationship indicators in fraud investigations because multiple applications originating from the same device may indicate coordinated fraudulent activity.

Since the BAF dataset does not contain device identifiers, synthetic DeviceIDs are generated using fraud-aware augmentation rules.

The generation process is guided by the `device_distinct_emails_8w` feature, which captures historical device-sharing behaviour. Fraudulent applications are assigned a higher probability of sharing DeviceIDs, while legitimate applications predominantly receive unique devices with limited sharing to reflect common scenarios such as shared family or organizational devices.

DeviceIDs are generated using a distributed PySpark DataFrame-based augmentation process. Fraudulent and legitimate applications are processed using configurable sharing probabilities, while fraud cluster sizes are controlled to simulate coordinated fraud rings. Spark DataFrame transformations are used throughout to ensure scalability and compatibility with Databricks Serverless compute.

Applicants assigned the same DeviceID retain the same operating system (`device_os`) to maintain realistic device consistency.

This approach produces investigation-ready device clusters suitable for graph-based fraud analysis.

In [0]:
# Device Augmentation Parameters

# Probability of reusing an existing device
FRAUD_DEVICE_SHARE_RATE = 0.70
LEGIT_DEVICE_SHARE_RATE = 0.08

# Device pools
FRAUD_DEVICE_POOL_SIZE = 2500
LEGIT_DEVICE_POOL_SIZE = 60000

# Fraud cluster size
MIN_FRAUD_CLUSTER_SIZE = 3
MAX_FRAUD_CLUSTER_SIZE = 8

## Phase 1: Generate Fraud Device Clusters
Fraudulent applications are assigned a higher probability of sharing DeviceIDs than legitimate applications. Shared fraud devices are generated with controlled cluster sizes to simulate coordinated fraud rings, while legitimate device sharing remains limited and reflects common real-world scenarios such as shared household or organizational devices.


In [0]:
# -------------------------------
# 1.1 Create DeviceID Pools
# -------------------------------

fraud_device_pool = [
    f"FDEV{i:06d}"
    for i in range(1, FRAUD_DEVICE_POOL_SIZE + 1)
]

legit_device_pool = [
    f"LDEV{i:06d}"
    for i in range(1, LEGIT_DEVICE_POOL_SIZE + 1)
]

In [0]:
print(f"Fraud Device Pool Size      : {len(fraud_device_pool)}")
print(f"Legitimate Device Pool Size : {len(legit_device_pool)}")

print("\nSample Fraud Devices:")
print(fraud_device_pool[:5])

print("\nSample Legitimate Devices:")
print(legit_device_pool[:5])

Fraud Device Pool Size      : 2500
Legitimate Device Pool Size : 60000

Sample Fraud Devices:
['FDEV000001', 'FDEV000002', 'FDEV000003', 'FDEV000004', 'FDEV000005']

Sample Legitimate Devices:
['LDEV000001', 'LDEV000002', 'LDEV000003', 'LDEV000004', 'LDEV000005']


### Inference

Two separate DeviceID pools have been created to model different sharing behaviours for fraudulent and legitimate applications. Fraud devices are intentionally reusable to simulate coordinated fraud patterns, while the larger legitimate device pool reflects that most genuine applicants use unique devices with only limited sharing. At this stage, DeviceIDs have not yet been assigned to any applicant.

In [0]:
from pyspark.sql import functions as F

# 1.2 Extract fraud applicants
fraud_df = (
    aug_df
    .filter(F.col("fraud_bool") == 1)
    .select("ApplicantID", "device_os", "device_distinct_emails_8w")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(fraud_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID,device_os,device_distinct_emails_8w
APP00000001,windows,1
APP00000148,linux,1
APP00000213,other,1
APP00000313,windows,1
APP00000619,windows,1
APP00000778,windows,1
APP00000806,windows,1
APP00000932,other,1
APP00000953,windows,1
APP00000972,windows,1


### Inference

Only fraudulent applications have been isolated for the first phase of DeviceID assignment. The extracted attributes provide the information required to generate realistic shared DeviceIDs while preserving operating system consistency and incorporating historical device-sharing behaviour through the `device_distinct_emails_8w` feature.

In [0]:
# 1.3 Create Sharing Priority
from pyspark.sql import functions as F

fraud_df = fraud_df.withColumn(
    "SharePriority",
    F.when(F.col("device_distinct_emails_8w") == 2, 3)
     .when(F.col("device_distinct_emails_8w") == 1, 2)
     .otherwise(1)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    fraud_df.groupBy(
        "device_distinct_emails_8w",
        "SharePriority"
    ).count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


device_distinct_emails_8w,SharePriority,count
1,2,9839
2,3,1035
0,1,151
-1,1,4


### Inference

A sharing priority has been assigned to each fraudulent application based on the historical device-sharing indicator (`device_distinct_emails_8w`). Applications with higher sharing priority will be more likely to receive shared DeviceIDs during augmentation, ensuring that the generated relationships reflect the behavioural patterns already present in the original dataset rather than relying solely on random assignment.

In [0]:
# 1.4 Create Fraud Clusters
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Randomize applicants
fraud_df = fraud_df.withColumn(
    "RandomOrder",
    F.rand(seed=42)
)

# Sort applicants randomly and assign sequence number
window_spec = Window.orderBy("RandomOrder")

# Assign FraudSequence
fraud_df = fraud_df.withColumn(
    "FraudSequence",
    F.row_number().over(window_spec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Inference

Fraudulent applications have been randomly distributed and assigned a unique sequence number. This randomized ordering removes any bias introduced by the original dataset order and provides the foundation for creating realistic fraud clusters in the subsequent step.

In [0]:
display(
    fraud_df
    .select(
        "ApplicantID",
        "RandomOrder",
        "FraudSequence"
    )
    .orderBy("FraudSequence")
    .limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID,RandomOrder,FraudSequence
APP00000619,4.295885923766285E-5,1
APP00744405,5.6749896668706334E-5,2
APP00505764,1.008142107472576E-4,3
APP00109660,1.5194336087298588E-4,4
APP00956259,1.5873277915567918E-4,5
APP00247268,3.0658315361520305E-4,6
APP00515803,5.130722128980914E-4,7
APP00519803,5.920131152373775E-4,8
APP00451174,6.801796515759628E-4,9
APP00368733,7.280062488664862E-4,10


In [0]:
# 1.5 Assign Fraud Cluster IDs

import random
from itertools import cycle
from pyspark.sql import functions as F

# Generate random cluster sizes between configured limits
cluster_sizes = []
remaining = fraud_df.count()

while remaining > 0:
    size = random.randint(MIN_FRAUD_CLUSTER_SIZE, MAX_FRAUD_CLUSTER_SIZE)

    # Prevent the last cluster from violating the minimum size
    if remaining - size < MIN_FRAUD_CLUSTER_SIZE and remaining - size != 0:
        size = remaining

    cluster_sizes.append(size)
    remaining -= size

# Create FraudClusterID list
cluster_ids = []

for cluster_no, size in enumerate(cluster_sizes, start=1):
    cluster_ids.extend([f"FCL{cluster_no:05d}"] * size)

# Create Spark DataFrame mapping sequence → cluster
cluster_map = spark.createDataFrame(
    list(enumerate(cluster_ids, start=1)),
    ["FraudSequence", "FraudClusterID"]
)

# Assign FraudClusterID
fraud_df = fraud_df.join(cluster_map, on="FraudSequence", how="left")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    fraud_df
    .groupBy("FraudClusterID")
    .agg(
        F.count("*").alias("ClusterSize")
    )
    .orderBy("FraudClusterID")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


FraudClusterID,ClusterSize
FCL00001,7
FCL00002,7
FCL00003,8
FCL00004,8
FCL00005,8
FCL00006,3
FCL00007,7
FCL00008,7
FCL00009,5
FCL00010,5


### Inference

Fraudulent applications have been grouped into variable-sized clusters according to the configured minimum and maximum cluster size limits. Each cluster is represented by a unique `FraudClusterID`, which will be used in the next step to assign a shared DeviceID. Separating cluster creation from DeviceID assignment improves traceability, validation, and maintainability of the augmentation process.

In [0]:
# 1.6 Assign Shared Fraud DeviceIDs

from pyspark.sql.window import Window
from pyspark.sql import functions as F

# Create one DeviceID for each FraudClusterID
device_map = (
    fraud_df
    .select("FraudClusterID")
    .distinct()
    .orderBy("FraudClusterID")
    .withColumn(
        "DeviceID",
        F.concat(
            F.lit("FDEV"),
            F.lpad(F.row_number().over(Window.orderBy("FraudClusterID")), 6, "0")
        )
    )
)

# Assign DeviceID to every applicant in the cluster
fraud_df = (
    fraud_df
    .join(device_map, on="FraudClusterID", how="left")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    fraud_df
    .groupBy("DeviceID")
    .count()
    .orderBy(F.desc("count"))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DeviceID,count
FDEV000468,8
FDEV001027,8
FDEV000814,8
FDEV000740,8
FDEV001116,8
FDEV000184,8
FDEV001041,8
FDEV001927,8
FDEV000984,8
FDEV001427,8


In [0]:
display(
    fraud_df
    .groupBy("DeviceID")
    .count()
    .groupBy("count")
    .count()
    .orderBy("count")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


count,count
3,314
4,352
5,328
6,330
7,341
8,334


### Inference

Fraud clusters have been successfully mapped to shared DeviceIDs. Each `FraudClusterID` corresponds to a single `DeviceID`, resulting in realistic many-to-one relationships between fraudulent applications and devices. The observed DeviceID sharing distribution matches the configured cluster size limits, confirming that the augmentation process has generated consistent fraud clusters suitable for graph-based analysis.

## Phase 2: Generate Legitimate Device Clusters

In [0]:
# -------------------------------
# 2.1 Extract Legitimate Applicants
# -------------------------------

from pyspark.sql import functions as F

# Extract legitimate applications
legit_df = (
    aug_df.filter(F.col("fraud_bool") == 0)
      .select(
          "ApplicantID",
          "device_os",
          "device_distinct_emails_8w"
      )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(legit_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID,device_os,device_distinct_emails_8w
APP00000002,other,0
APP00000003,windows,1
APP00000004,other,1
APP00000005,x11,1
APP00000006,other,1
APP00000007,other,1
APP00000008,other,1
APP00000009,windows,1
APP00000010,windows,1
APP00000011,other,1


### Inference

Legitimate applications have been extracted for DeviceID generation. These records will be used to assign mostly unique DeviceIDs with limited controlled sharing.

In [0]:
# -------------------------------
# 2.2 Create Sharing Priority
# -------------------------------

from pyspark.sql.window import Window
from pyspark.sql import functions as F

window_spec = Window.orderBy(
    F.desc("device_distinct_emails_8w"),
    F.rand(seed=42)
)

legit_df = legit_df.withColumn(
    "SharingPriority",
    F.row_number().over(window_spec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    legit_df
    .select(
        "ApplicantID",
        "device_distinct_emails_8w",
        "SharingPriority"
    )
    .orderBy("SharingPriority")
    .limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID,device_distinct_emails_8w,SharingPriority
APP00866490,2,1
APP00301882,2,2
APP00821789,2,3
APP00641583,2,4
APP00263653,2,5
APP00298129,2,6
APP00893805,2,7
APP00307487,2,8
APP00289284,2,9
APP00145337,2,10


### Inference

Legitimate applicants have been prioritized for DeviceID sharing based on recent device usage, with ties resolved randomly.

In [0]:
# -------------------------------
# 2.3 Separate Shared and Unique Applicants
# -------------------------------

shared_count = int(legit_df.count() * LEGIT_DEVICE_SHARE_RATE)

# Applicants selected for shared DeviceIDs
shared_legit_df = (
    legit_df
    .filter(F.col("SharingPriority") <= shared_count)
)

# Remaining applicants receive unique DeviceIDs
unique_legit_df = (
    legit_df
    .filter(F.col("SharingPriority") > shared_count)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    shared_legit_df.select(F.count("*").alias("SharedApplicants"))
)

display(
    unique_legit_df.select(F.count("*").alias("UniqueApplicants"))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


SharedApplicants
79117


UniqueApplicants
909854


### Inference

Legitimate applicants have been separated into shared and unique groups according to the configured sharing rate.

In [0]:
# -------------------------------
# 2.4 Create Legitimate Device Clusters
# -------------------------------

from pyspark.sql.window import Window
from pyspark.sql import functions as F

shared_legit_df = shared_legit_df.withColumn(
    "RandomOrder",
    F.rand(seed=42)
)

window_spec = Window.orderBy("RandomOrder")

shared_legit_df = shared_legit_df.withColumn(
    "LegitSequence",
    F.row_number().over(window_spec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    shared_legit_df
    .select(
        "ApplicantID",
        "RandomOrder",
        "LegitSequence"
    )
    .orderBy("LegitSequence")
    .limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID,RandomOrder,LegitSequence
APP00821083,4.126290905404062E-6,1
APP00045190,4.135636149316113E-6,2
APP00778994,5.07959462925367E-6,3
APP00806770,6.209747072882799E-6,4
APP00065067,1.2430797129203697E-5,5
APP00874604,2.4556884532067613E-5,6
APP00681193,3.1183656878597255E-5,7
APP00263653,4.295885923766285E-5,8
APP00383383,5.6749896668706334E-5,9
APP00199005,8.919584367883537E-5,10


### Inference

Shared legitimate applicants have been randomly ordered to prepare for cluster creation.

In [0]:
MIN_LEGIT_CLUSTER_SIZE = 2
MAX_LEGIT_CLUSTER_SIZE = 4

# -------------------------------
# 2.5 Assign Legitimate Cluster IDs
# -------------------------------

import random

cluster_sizes = []
remaining = shared_legit_df.count()

while remaining > 0:
    size = random.randint(
        MIN_LEGIT_CLUSTER_SIZE,
        MAX_LEGIT_CLUSTER_SIZE
    )

    if remaining - size < MIN_LEGIT_CLUSTER_SIZE and remaining - size != 0:
        size = remaining

    cluster_sizes.append(size)
    remaining -= size

cluster_ids = []

for cluster_no, size in enumerate(cluster_sizes, start=1):
    cluster_ids.extend([f"LCL{cluster_no:05d}"] * size)

cluster_map = spark.createDataFrame(
    list(enumerate(cluster_ids, start=1)),
    ["LegitSequence", "LegitClusterID"]
)

shared_legit_df = shared_legit_df.join(
    cluster_map,
    on="LegitSequence",
    how="left"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    shared_legit_df
    .groupBy("LegitClusterID")
    .agg(F.count("*").alias("ClusterSize"))
    .orderBy("LegitClusterID")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


LegitClusterID,ClusterSize
LCL00001,4
LCL00002,4
LCL00003,4
LCL00004,3
LCL00005,2
LCL00006,4
LCL00007,2
LCL00008,4
LCL00009,2
LCL00010,4


### Inference

Shared legitimate applicants have been grouped into small clusters for controlled DeviceID sharing.

In [0]:
# -------------------------------
# 2.6 Assign Shared DeviceIDs
# -------------------------------

from pyspark.sql.window import Window
from pyspark.sql import functions as F

device_map = (
    shared_legit_df
    .select("LegitClusterID")
    .distinct()
    .orderBy("LegitClusterID")
    .withColumn(
        "DeviceID",
        F.concat(
            F.lit("LDEV"),
            F.lpad(
                F.row_number().over(Window.orderBy("LegitClusterID")),
                6,
                "0"
            )
        )
    )
)

shared_legit_df = shared_legit_df.join(
    device_map,
    on="LegitClusterID",
    how="left"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    shared_legit_df
    .groupBy("DeviceID")
    .count()
    .orderBy(F.desc("count"))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DeviceID,count
LDEV026075,4
LDEV009955,4
LDEV025534,4
LDEV019596,4
LDEV024044,4
LDEV018332,4
LDEV018426,4
LDEV025092,4
LDEV015357,4
LDEV017237,4


### Inference

Shared legitimate clusters have been assigned common DeviceIDs, creating realistic many-to-one relationships between applicants and devices.

In [0]:
# -------------------------------
# 2.7 Assign Unique DeviceIDs
# -------------------------------

from pyspark.sql.window import Window
from pyspark.sql import functions as F

shared_devices = device_map.count()

unique_legit_df = unique_legit_df.withColumn(
    "DeviceID",
    F.concat(
        F.lit("LDEV"),
        F.lpad(
            F.row_number().over(Window.orderBy("SharingPriority")) + shared_devices,
            6,
            "0"
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    unique_legit_df
    .select(
        "ApplicantID",
        "DeviceID"
    )
    .limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ApplicantID,DeviceID
APP00353121,LDEV026444
APP00805847,LDEV026445
APP00473950,LDEV026446
APP00577967,LDEV026447
APP00441346,LDEV026448
APP00726537,LDEV026449
APP00657694,LDEV026450
APP00314285,LDEV026451
APP00387890,LDEV026452
APP00863163,LDEV026453


### Inference

Unique DeviceIDs have been assigned to legitimate applicants not selected for device sharing.

In [0]:
# -------------------------------
# 2.8 Merge Legitimate Device Assignments
# -------------------------------

legit_device_df = (
    shared_legit_df.select("ApplicantID", "DeviceID")
    .unionByName(
        unique_legit_df.select("ApplicantID", "DeviceID")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    legit_device_df.select(
        F.count("*").alias("TotalApplicants"),
        F.countDistinct("ApplicantID").alias("UniqueApplicants"),
        F.countDistinct("DeviceID").alias("UniqueDeviceIDs")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


TotalApplicants,UniqueApplicants,UniqueDeviceIDs
988971,988971,936297


### Inference

Shared and unique DeviceID assignments have been merged into a single dataset for all legitimate applicants.

In [0]:
shared_legit_df.printSchema()
unique_legit_df.printSchema()
legit_device_df.printSchema()

root
 |-- LegitClusterID: string (nullable = true)
 |-- LegitSequence: integer (nullable = false)
 |-- ApplicantID: string (nullable = false)
 |-- device_os: string (nullable = true)
 |-- device_distinct_emails_8w: integer (nullable = true)
 |-- SharingPriority: integer (nullable = false)
 |-- RandomOrder: double (nullable = false)
 |-- DeviceID: string (nullable = true)

root
 |-- ApplicantID: string (nullable = false)
 |-- device_os: string (nullable = true)
 |-- device_distinct_emails_8w: integer (nullable = true)
 |-- SharingPriority: integer (nullable = false)
 |-- DeviceID: string (nullable = false)

root
 |-- ApplicantID: string (nullable = false)
 |-- DeviceID: string (nullable = true)



## Phase 3: Attach DeviceIDs to Applications


In [0]:
# -------------------------------
# 3.1 Merge Fraud and Legitimate Device Assignments
# -------------------------------

device_df = (
    fraud_df
    .select("ApplicantID", "DeviceID")
    .unionByName(legit_device_df)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    device_df.select(
        F.count("*").alias("TotalApplicants"),
        F.countDistinct("ApplicantID").alias("UniqueApplicants"),
        F.countDistinct("DeviceID").alias("UniqueDeviceIDs")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


TotalApplicants,UniqueApplicants,UniqueDeviceIDs
1000000,1000000,938296


### Inference

Fraud and legitimate DeviceID assignments have been combined into a single mapping for all applicants.

In [0]:
# -------------------------------
# 3.2 Attach DeviceIDs to Applications
# -------------------------------

aug_df = (
    aug_df
    .join(
        device_df,
        on="ApplicantID",
        how="left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Inference

DeviceIDs have been successfully attached to all applications, making the dataset ready for IP address generation.

# Phase 4: Generate IP Address Relationships

In [0]:
# -------------------------------
# 4.1 Configure IP Generation
# -------------------------------

FRAUD_IP_POOL_SIZE = 5000
LEGIT_IP_POOL_SIZE = 1000000

FRAUD_IP_SHARE_RATE = 0.70
LEGIT_IP_SHARE_RATE = 0.05

MIN_FRAUD_IP_CLUSTER_SIZE = 3
MAX_FRAUD_IP_CLUSTER_SIZE = 8

MIN_LEGIT_IP_CLUSTER_SIZE = 2
MAX_LEGIT_IP_CLUSTER_SIZE = 4

In [0]:
print(f"Fraud IP Pool Size      : {FRAUD_IP_POOL_SIZE}")
print(f"Legitimate IP Pool Size : {LEGIT_IP_POOL_SIZE}")
print(f"Fraud IP Share Rate     : {FRAUD_IP_SHARE_RATE:.0%}")
print(f"Legit IP Share Rate     : {LEGIT_IP_SHARE_RATE:.0%}")

Fraud IP Pool Size      : 5000
Legitimate IP Pool Size : 1000000
Fraud IP Share Rate     : 70%
Legit IP Share Rate     : 5%


### Inference

The IP address generation parameters have been configured for fraud and legitimate applications.

In [0]:
# -------------------------------
# 4.2 Generate Fraud IP Pool
# -------------------------------

fraud_ip_pool = [
    f"10.{i // (256 * 254)}.{(i // 254) % 256}.{(i % 254) + 1}"
    for i in range(FRAUD_IP_POOL_SIZE)
]

In [0]:
print(f"Fraud IPs Generated : {len(fraud_ip_pool)}")
print(f"Unique Fraud IPs    : {len(set(fraud_ip_pool))}")

fraud_ip_pool[:5]

Fraud IPs Generated : 5000
Unique Fraud IPs    : 5000


['10.0.0.1', '10.0.0.2', '10.0.0.3', '10.0.0.4', '10.0.0.5']

### Inference

A pool of synthetic private IPv4 addresses has been generated for fraudulent applications.

In [0]:
# -------------------------------
# 4.3 Generate Legitimate IP Pool
# -------------------------------

legit_ip_pool = [
    f"172.{16 + (i // (256 * 254))}.{(i // 254) % 256}.{(i % 254) + 1}"
    for i in range(LEGIT_IP_POOL_SIZE)
]

In [0]:
print(f"Legitimate IPs Generated : {len(legit_ip_pool)}")
print(f"Unique Legitimate IPs    : {len(set(legit_ip_pool))}")

legit_ip_pool[:5]

Legitimate IPs Generated : 1000000
Unique Legitimate IPs    : 1000000


['172.16.0.1', '172.16.0.2', '172.16.0.3', '172.16.0.4', '172.16.0.5']

### Inference

A pool of synthetic private IPv4 addresses has been generated for legitimate applications.

In [0]:
# -------------------------------
# 4.4 Convert IP Pools to Spark DataFrames
# -------------------------------

fraud_ip_pool_df = spark.createDataFrame(
    [(i + 1, ip) for i, ip in enumerate(fraud_ip_pool)],
    ["IPIndex", "IPAddress"]
)

legit_ip_pool_df = spark.createDataFrame(
    [(i + 1, ip) for i, ip in enumerate(legit_ip_pool)],
    ["IPIndex", "IPAddress"]
)

In [0]:
print(f"Fraud IP Pool Size : {fraud_ip_pool_df.count():,}")
print(f"Legit IP Pool Size : {legit_ip_pool_df.count():,}")

print(f"Unique Fraud IPs : {fraud_ip_pool_df.select('IPAddress').distinct().count():,}")
print(f"Unique Legit IPs : {legit_ip_pool_df.select('IPAddress').distinct().count():,}")

fraud_ip_pool_df.show(5, truncate=False)
legit_ip_pool_df.show(5, truncate=False)

Fraud IP Pool Size : 5,000
Legit IP Pool Size : 1,000,000
Unique Fraud IPs : 5,000
Unique Legit IPs : 1,000,000
+-------+---------+
|IPIndex|IPAddress|
+-------+---------+
|1      |10.0.0.1 |
|2      |10.0.0.2 |
|3      |10.0.0.3 |
|4      |10.0.0.4 |
|5      |10.0.0.5 |
+-------+---------+
only showing top 5 rows
+-------+----------+
|IPIndex|IPAddress |
+-------+----------+
|1      |172.16.0.1|
|2      |172.16.0.2|
|3      |172.16.0.3|
|4      |172.16.0.4|
|5      |172.16.0.5|
+-------+----------+
only showing top 5 rows


### Inference

Fraud and legitimate IP pools have been converted into Spark DataFrames with indexed IP addresses, enabling efficient and deterministic IP assignment in subsequent steps.

In [0]:
# -------------------------------
# 4.5 Extract Fraud and Legitimate Applications
# -------------------------------

fraud_ip_df = aug_df.filter(F.col("fraud_bool") == 1)

legit_ip_df = aug_df.filter(F.col("fraud_bool") == 0)

In [0]:
# -------------------------------
# Validation - Step 4.5
# -------------------------------

print(f"Fraud Applications : {fraud_ip_df.count():,}")
print(f"Legit Applications : {legit_ip_df.count():,}")
print(f"Total Applications : {fraud_ip_df.count() + legit_ip_df.count():,}")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Fraud Applications : 11,029
Legit Applications : 988,971
Total Applications : 1,000,000


### Inference

Fraudulent and legitimate applications have been separated to enable independent IP address assignment strategies based on their respective sharing patterns.

In [0]:
# -------------------------------
# 4.6 Assign Share Priority
# -------------------------------

window_spec = Window.orderBy("ApplicantID")

fraud_ip_df = (
    fraud_ip_df
    .withColumn("SharePriority", F.row_number().over(window_spec))
)

legit_ip_df = (
    legit_ip_df
    .withColumn("SharePriority", F.row_number().over(window_spec))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# -------------------------------
# Validation - Step 4.6
# -------------------------------

print(f"Fraud Records : {fraud_ip_df.count():,}")
print(f"Legit Records : {legit_ip_df.count():,}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Fraud Records : 11,029
Legit Records : 988,971


In [0]:
print(fraud_ip_df.columns)
print(legit_ip_df.columns)

['ApplicantID', 'fraud_bool', 'income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'employment_status', 'credit_risk_score', 'email_is_free', 'housing_status', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'source', 'session_length_in_minutes', 'device_os', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'month', 'is_false_positive_alert', 'GenerateAlert', 'AlertID', 'DeviceID', 'SharePriority']
['ApplicantID', 'fraud_bool', 'income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'payment_type', 'zip_count_4w', 'velocity_6h', 'velocity_24h'

### Inference

A sequential share priority has been assigned to fraud and legitimate applications to support deterministic allocation of shared and unique IP addresses in the subsequent steps.

In [0]:
# -------------------------------
# 4.7 Split into Shared and Unique Applications
# -------------------------------

fraud_shared_count = int(fraud_ip_df.count() * FRAUD_IP_SHARE_RATE)
legit_shared_count = int(legit_ip_df.count() * LEGIT_IP_SHARE_RATE)

shared_fraud_df = fraud_ip_df.filter(F.col("SharePriority") <= fraud_shared_count)
unique_fraud_df = fraud_ip_df.filter(F.col("SharePriority") > fraud_shared_count)

shared_legit_df = legit_ip_df.filter(F.col("SharePriority") <= legit_shared_count)
unique_legit_df = legit_ip_df.filter(F.col("SharePriority") > legit_shared_count)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# -------------------------------
# Validation - Step 4.7
# -------------------------------

print(f"Shared Fraud Applications : {shared_fraud_df.count():,}")
print(f"Unique Fraud Applications : {unique_fraud_df.count():,}")

print(f"Shared Legit Applications : {shared_legit_df.count():,}")
print(f"Unique Legit Applications : {unique_legit_df.count():,}")

print(f"Total Fraud : {shared_fraud_df.count() + unique_fraud_df.count():,}")
print(f"Total Legit : {shared_legit_df.count() + unique_legit_df.count():,}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Shared Fraud Applications : 7,720
Unique Fraud Applications : 3,309
Shared Legit Applications : 49,448
Unique Legit Applications : 939,523
Total Fraud : 11,029
Total Legit : 988,971


### Inference

Fraud and legitimate applications have been divided into shared and unique groups according to the configured sharing rates, preparing them for differentiated IP address assignment.

In [0]:
# -------------------------------
# 4.8 Assign Fraud IP Addresses
# -------------------------------

shared_fraud_with_cluster = (
    shared_fraud_df
    .withColumn(
        "FraudIPClusterID",
        (
            (
                F.col("SharePriority") - 1
            ) / (
                (MIN_FRAUD_IP_CLUSTER_SIZE + MAX_FRAUD_IP_CLUSTER_SIZE) // 2
            )
        ).cast("int") + 1
    )
)

shared_fraud_ip_df = (
    shared_fraud_with_cluster
    .join(
        fraud_ip_pool_df.withColumnRenamed("IPIndex", "FraudIPClusterID"),
        "FraudIPClusterID",
        "left"
    )
)

unique_fraud_ip_df = (
    unique_fraud_df
    .withColumn(
        "IPIndex",
        F.row_number().over(Window.orderBy("SharePriority"))
    )
    .join(
        fraud_ip_pool_df,
        "IPIndex",
        "left"
    )
)

fraud_ip_df = (
    shared_fraud_ip_df
    .select("ApplicantID", "IPAddress")
    .unionByName(
        unique_fraud_ip_df.select("ApplicantID", "IPAddress")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# -------------------------------
# Validation - Step 4.8
# -------------------------------

print(f"Fraud Applicants Assigned IPs : {fraud_ip_df.count():,}")
print(f"Unique Fraud IPs Used : {fraud_ip_df.select('IPAddress').distinct().count():,}")

fraud_ip_df.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Fraud Applicants Assigned IPs : 11,029
Unique Fraud IPs Used : 3,309
+-----------+---------+
|ApplicantID|IPAddress|
+-----------+---------+
|APP00000134|10.0.0.1 |
|APP00000148|10.0.0.1 |
|APP00000185|10.0.0.1 |
|APP00000195|10.0.0.1 |
|APP00000341|10.0.0.1 |
+-----------+---------+
only showing top 5 rows


### Inference

Fraudulent applications have been assigned IP addresses using a combination of shared IP clusters and unique IPs, simulating coordinated fraud behaviour while preserving individual records.

In [0]:
# -------------------------------
# 4.9 Assign Legitimate IP Addresses
# -------------------------------

shared_legit_with_cluster = (
    shared_legit_df
    .withColumn(
        "LegitIPClusterID",
        (
            (
                F.col("SharePriority") - 1
            ) / (
                (MIN_LEGIT_IP_CLUSTER_SIZE + MAX_LEGIT_IP_CLUSTER_SIZE) // 2
            )
        ).cast("int") + 1
    )
)

shared_legit_ip_df = (
    shared_legit_with_cluster
    .join(
        legit_ip_pool_df.withColumnRenamed("IPIndex", "LegitIPClusterID"),
        "LegitIPClusterID",
        "left"
    )
)

unique_legit_ip_df = (
    unique_legit_df
    .withColumn(
        "IPIndex",
        F.row_number().over(Window.orderBy("SharePriority"))
    )
    .join(
        legit_ip_pool_df,
        "IPIndex",
        "left"
    )
)

legit_ip_df = (
    shared_legit_ip_df
    .select("ApplicantID", "IPAddress")
    .unionByName(
        unique_legit_ip_df.select("ApplicantID", "IPAddress")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# -------------------------------
# Validation - Step 4.9
# -------------------------------

print(f"Legit Applicants Assigned IPs : {legit_ip_df.count():,}")
print(f"Unique Legit IPs Used : {legit_ip_df.select('IPAddress').distinct().count():,}")

legit_ip_df.show(5, truncate=False)

legit_ip_df.filter(F.col("IPAddress").isNull()).count()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Legit Applicants Assigned IPs : 988,971
Unique Legit IPs Used : 939,523
+-----------+----------+
|ApplicantID|IPAddress |
+-----------+----------+
|APP00000001|172.16.0.1|
|APP00000002|172.16.0.1|
|APP00000003|172.16.0.1|
|APP00000004|172.16.0.2|
|APP00000005|172.16.0.2|
+-----------+----------+
only showing top 5 rows


0

### Inference

Legitimate applications have been assigned IP addresses using a mix of shared and unique IPs, reflecting realistic usage patterns where limited IP sharing occurs among genuine users.

In [0]:
# -------------------------------
# 4.10 Merge Fraud and Legitimate IP Assignments
# -------------------------------

ip_df = fraud_ip_df.unionByName(legit_ip_df)

In [0]:
# -------------------------------
# Validation - Step 4.10
# -------------------------------

print(f"Total Applicants Assigned IPs : {ip_df.count():,}")
print(f"Unique IP Addresses : {ip_df.select('IPAddress').distinct().count():,}")

print(ip_df.select("ApplicantID").distinct().count())
print(ip_df.filter(F.col("IPAddress").isNull()).count())
ip_df.show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total Applicants Assigned IPs : 1,000,000


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Unique IP Addresses : 942,832
1000000
0
+-----------+---------+
|ApplicantID|IPAddress|
+-----------+---------+
|APP00000134|10.0.0.1 |
|APP00000148|10.0.0.1 |
|APP00000185|10.0.0.1 |
|APP00000195|10.0.0.1 |
|APP00000341|10.0.0.1 |
+-----------+---------+
only showing top 5 rows


### Inference

Fraudulent and legitimate IP assignments have been combined into a single dataset, providing one IP address for every applicant while preserving the intended IP sharing patterns.

In [0]:
# -------------------------------
# 4.11 Join IP Addresses with Dataset
# -------------------------------

aug_df = (
    aug_df
    .join(ip_df, on="ApplicantID", how="left")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# -------------------------------
# Validation - Step 4.11
# -------------------------------

print(f"Total Records : {aug_df.count():,}")
print(f"Missing IP Addresses : {aug_df.filter(F.col('IPAddress').isNull()).count():,}")

aug_df.select(
    "ApplicantID",
    "fraud_bool",
    "IPAddress"
).show(5, truncate=False)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total Records : 1,000,000
Missing IP Addresses : 0
+-----------+----------+----------+
|ApplicantID|fraud_bool|IPAddress |
+-----------+----------+----------+
|APP00000006|0         |172.16.0.2|
|APP00000003|0         |172.16.0.1|
|APP00000004|0         |172.16.0.2|
|APP00000002|0         |172.16.0.1|
|APP00000005|0         |172.16.0.2|
+-----------+----------+----------+
only showing top 5 rows


### Inference

The assigned IP addresses have been successfully integrated into the applicant dataset, ensuring every application is associated with an IP address for downstream graph construction and risk analysis.

### Generate RegistrationTimestamp

In [0]:
from pyspark.sql import functions as F

# Legitimate applications: spread across 180 days
legit_start = F.unix_timestamp(F.lit("2025-01-01 00:00:00"))
legit_end = F.unix_timestamp(F.lit("2025-06-30 23:59:59"))

# Fraud campaign windows
campaign_1 = F.unix_timestamp(F.lit("2025-02-10 08:00:00"))
campaign_2 = F.unix_timestamp(F.lit("2025-04-18 14:00:00"))
campaign_3 = F.unix_timestamp(F.lit("2025-06-05 20:00:00"))

# Random campaign selection
campaign = (
    F.when(F.rand(11) < 0.33, campaign_1)
     .when(F.rand(22) < 0.66, campaign_2)
     .otherwise(campaign_3)
)

aug_df = (
    aug_df.withColumn(
        "RegistrationTimestamp",
        F.when(
            F.col("fraud_bool") == 0,
            F.from_unixtime(
                legit_start +
                (F.rand(1) * (legit_end - legit_start)).cast("long")
            ).cast("timestamp")
        ).otherwise(
            F.from_unixtime(
                campaign +
                (F.rand(2) * 7200).cast("long")   # within 2 hours
            ).cast("timestamp")
        )
    )
)

In [0]:
aug_df.select(
    F.min("RegistrationTimestamp").alias("Earliest"),
    F.max("RegistrationTimestamp").alias("Latest")
).show(truncate=False)

aug_df.select("RegistrationTimestamp").show(5, truncate=False)

+-------------------+-------------------+
|Earliest           |Latest             |
+-------------------+-------------------+
|2025-01-01 00:00:33|2025-06-30 23:59:52|
+-------------------+-------------------+

+---------------------+
|RegistrationTimestamp|
+---------------------+
|2025-01-02 23:26:23  |
|2025-06-10 08:34:31  |
|2025-01-29 16:17:19  |
|2025-05-11 15:35:10  |
|2025-03-04 21:42:46  |
+---------------------+
only showing top 5 rows


In [0]:
aug_df.filter(F.col("fraud_bool") == 1)\
      .groupBy(F.date_format("RegistrationTimestamp", "yyyy-MM-dd").alias("Date"))\
      .count()\
      .orderBy(F.col("count").desc())\
      .show(10)

+----------+-----+
|      Date|count|
+----------+-----+
|2025-04-18| 4916|
|2025-02-10| 3609|
|2025-06-05| 2504|
+----------+-----+



### Inference

Registration timestamps were successfully generated using realistic temporal patterns. Legitimate applications are distributed across the observation period, while fraudulent applications are concentrated into campaign bursts. The updated dataset has been persisted to the Delta table for consistent use across PostgreSQL, Neo4j, and ML pipelines.

In [0]:
# Save Delta Table
(
    aug_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("risknet_augmented_dataset")
)

### Inference

The augmented dataset has been successfully loaded from the Delta table and is ready for graph node and relationship generation.